# Interactive Lab: Prompt Specification & Schema Validator

Welcome to the interactive workbook for **Module 3: Prompt to Specification**. 

This notebook will guide you through loading a raw system change note, validating inputs, executing a structured call to the Gemini API, and enforcing a strict Pydantic output schema to prevent formatting defects.

## 🛠️ Step 1: Install & Import Dependencies

First, we make sure our required packages are imported. Ensure you have loaded your environment variables (especially `GEMINI_API_KEY`).

In [ ]:
import os
from typing import List, Optional
from pydantic import BaseModel, Field
import google.generativeai as genai
from dotenv import load_dotenv

# Load local .env credentials if present
load_dotenv()

## 📐 Step 2: Define the Output Schema (Pydantic Contract)

We define the strict output schema that the model *must* return. This corresponds to the **Output Schema** defined in [Template 05: Prompt Specification Template](../templates/prompt-specification.md).

In [ ]:
class ActionStep(BaseModel):
    step_number: int = Field(description="Sequential step number starting at 1")
    action: str = Field(description="Action description in clear, non-technical words")
    deadline: Optional[str] = Field(None, description="ISO 8601 formatted date if applicable, otherwise omit")

class ChangeAnnouncementSchema(BaseModel):
    announcement_headline: str = Field(description="Short, descriptive headline containing system name")
    impact_summary: str = Field(description="Brief summary of user impact, free of developer jargon")
    key_actions_required: List[ActionStep] = Field(description="Actions target audience must take")
    technical_details_retained: str = Field(description="Underlying engineering detail retained for reference")
    support_contact: str = Field(default="support@company.com", description="Point of contact")

## 📝 Step 3: Input Variables & The Grounding Prompt Spec

Now we pass our raw developer commits and audience details into the Prompt Specification instructions.

In [ ]:
# Dynamic Input Variables
raw_change_notes = "Migrated session DB to PostgreSQL cluster. UI login screen will experience intermittent failures for 15 minutes at 3 AM GMT. QA validated."
target_audience = "External customer portal users"
date_effective = "2026-06-20"
system_impact = "Login UI"

# Formulate Prompt Specification with strict instructions
prompt_instructions = f"""
You are an expert change communications specialist. Your goal is to draft clear, audience-appropriate notifications based on technical notes.

Follow these reasoning guidelines:
1. Do not introduce any external system details not mentioned in the notes.
2. Do not assume policy rules; if instructions are unclear, write a generic warning.
3. Translate technical terms (e.g., 'PostgreSQL cluster') into simple impact summaries ('standard database migration').
4. Retain key technical notes in the 'technical_details_retained' field.

Input variables to process:
- Raw change notes: {raw_change_notes}
- Target audience: {target_audience}
- Date effective: {date_effective}
- System impact: {system_impact}
"""

## 🤖 Step 4: Call Gemini & Enforce Schema Output

We initialize our generative model and use the `response_schema` parameter to guarantee that the model outputs valid JSON conforming to our Pydantic model contract.

In [ ]:
# Initialize Gemini API
api_key = os.environ.get("GEMINI_API_KEY")

if not api_key:
    print("❌ ERROR: GEMINI_API_KEY environment variable not set. Please set it to run validation!")
else:
    genai.configure(api_key=api_key)
    
    # We use gemini-1.5-pro or gemini-2.0-flash which support structured JSON output schemas
    model = genai.GenerativeModel('gemini-1.5-pro')
    
    print("Sending prompt with response schema validation...")
    response = model.generate_content(
        prompt_instructions,
        generation_config=genai.types.GenerationConfig(
            response_mime_type="application/json",
            response_schema=ChangeAnnouncementSchema,
            temperature=0.1
        )
    )
    
    # Parse output using our Pydantic class to verify compatibility
    validated_output = ChangeAnnouncementSchema.model_validate_json(response.text)
    print("\n🎉 Schema Validation Complete!")
    print("Validated JSON Output:\n")
    print(validated_output.model_dump_json(indent=2))